In [ ]:
import json
import duckdb
import requests

In [ ]:
db_connection = duckdb.connect('loadsmart/dev.duckdb')

In [ ]:
manifest = 'loadsmart/target/manifest.json'
with open(manifest, 'r', encoding='utf-8') as f:
    manifest = json.load(f)

In [ ]:
table_content = []
for item_id, item in manifest.get('nodes', {}).items():
    if item.get('resource_type') == 'model':
        table = item.get('name')
        table_ds = item.get('description', '')
        columns = item.get('columns', {})
        columns_ds = [f" - {col_name}: {col_info.get('description', '')}" for col_name, col_info in columns.items()]
        table_content.append(f"Table: {table}\nDescription: {table_ds}\nColumns:\n" + "\n".join(columns_ds))

In [ ]:
ai_guide = "\n\n".join(table_content)

In [ ]:
def ask_ai_db_question(question):
    ai_prompt = f"""
        You are an expert DuckDB SQL AI.

        SCHEMA METADATA (Generated from dbt):
        {ai_guide}

        CRITICAL BEHAVIORAL RULES:
        1. You MUST read and strictly obey all `description` fields in the schema metadata above. They contain mandatory rules for handling historical dates, avoiding CURRENT_DATE, and filtering metrics.
        2. Write a valid DuckDB SQL query to answer the question.
        3. Return ONLY the executable SQL query in a markdown code block (```sql ... ```).

        Question: {question}
    """

    api_response = requests.post(
        "http://localhost:11434/api/generate",
        json = {
            'model': 'qwen2.5-coder:1.5b',
            'prompt': ai_prompt,
            'stream': False,
            'options': {'temperature': 0.0}
        }
    )

    llm_response = api_response.json()
    raw_sql = llm_response.get('response', '')
    cleaned_sql = raw_sql.replace('```sql', '').replace('```', '').strip()

    print(f"--- SQL ---\n{cleaned_sql}\n ---")

    result_table = db_connection.execute(cleaned_sql).fetchdf()
    return cleaned_sql, result_table

In [ ]:
ai_question = "How many loads were delivered in the last full month available in the data?"
sql_query, result_df = ask_ai_db_question(ai_question)
display(result_df)

--- SQL ---

 ---


AttributeError: 'NoneType' object has no attribute 'fetchdf'